# 06 · Build Core

**What this does:** turns the newest raw revenue and media into the clean daily tables the MMM uses, in the BigQuery dataset `us_hment_mmm_core`. Each step checks its totals against the raw data.

| | |
|---|---|
| **When to run** | After new revenue (01) or new media (02) is loaded, and after saving title changes in 05_Review_Titles |
| **How long** | About 1–2 minutes |
| **What it changes** | Only when `DRY_RUN = False`: rebuilds `revenue_film_daily` (with the view `revenue_premium_daily`) and `media_prisma_daily`. Raw data and film details are never changed. |
| **Safe to re-run?** | Yes. Each run rebuilds the tables from scratch from the newest data. |

### How to use it
1. Leave `DRY_RUN = True` and click **Run ▸ Run All Cells**. Read the totals in Steps 2 and 3. Nothing is written.
2. If they look right, set `DRY_RUN = False` and **Run All** again. Each step should say **✅ SAVED and checked**.
3. Set `DRY_RUN = True` again afterwards.

### What the steps use
| Step | Table | Rules |
|---|---|---|
| 2 · Revenue | `revenue_film_daily`, view `revenue_premium_daily` | US royalty only; Amazon from "Amazon Royalty" (not "Amazon Sale"); linked to films by revenue product ID. The view is the premium window (PVOD up to the day before EST) with pre-sales added to the PVOD day. |
| 3 · Media (Prisma) | `media_prisma_daily` | Advertisers ZUE, ZFF, ZFE; approved campaign budgets only; no deleted, Fee or test buys. Each buy month is spread evenly over the days of the placement's flight in that month. Linked to films by product code; non-film campaigns are kept but not added to any film. |

TV, audio and MVPD spend (NetPak / SpotPak) and channel labels will be added to this notebook next.

## Step 1 · Settings

In [1]:
# True  = check only, nothing is written (always start here)
# False = rebuild the core tables
DRY_RUN = True

## Step 2 · Revenue
*Nothing to edit here.*

In [2]:
from IPython.display import Markdown, display

from us_hment_mmm.core_display import show_result
from us_hment_mmm.revenue.build_core import run as build_revenue

try:
    revenue = build_revenue(dry_run=DRY_RUN)
except Exception as exc:
    display(Markdown(f"### 🛑 STOP — core revenue could not be built\n\n`{exc}`\n\nNothing was "
                     "changed by this step. Save this notebook and send it to the pipeline owner."))
    raise

show_result(revenue)

### 🔎 CHECK ONLY — nothing was written

Nothing was written. To save, set DRY_RUN = False and run again.

Revenue delivery **2026-10-30** · US rows only · left out: Amazon Sale · 324 films have revenue product IDs (324 with revenue in this delivery)

**Royalty**

Item,Royalty
Royalty kept from the delivery,"$2,308,278,113"
Linked to a film,"$1,435,962,546"
↳ sold before PVOD (pre-sales),"$15,169,884"
↳ premium window (PVOD to EST − 1),"$848,906,629"
↳ on or after EST,"$430,138,954"
↳ film has no PVOD date,"$141,747,079"
Used by the MMM (pre-sales + premium),"$864,076,513"


**Notes**

- 159 film(s) have revenue but no PVOD date. Their revenue is kept in the table but not used by the MMM until a date is added in 05_Review_Titles. In MMM = yes: 0 ($0); In MMM = no: 1; not reviewed: 158; released before March 2020 (before PVOD existed): 120.

## Step 3 · Media (Prisma)
*Nothing to edit here.*

In [3]:
from IPython.display import Markdown, display

from us_hment_mmm.core_display import show_result
from us_hment_mmm.media.build_core import run as build_media

try:
    media = build_media(dry_run=DRY_RUN)
except Exception as exc:
    display(Markdown(f"### 🛑 STOP — core media could not be built\n\n`{exc}`\n\nNothing was "
                     "changed by this step. Save this notebook and send it to the pipeline owner."))
    raise

show_result(media)

### 🔎 CHECK ONLY — nothing was written

Nothing was written. To save, set DRY_RUN = False and run again.

Prisma snapshot **2026-10-30** · advertisers ZUE, ZFF, ZFE · approved campaign budgets only

**Spend kept**

Item,Spend,Buy rows
Linked to a film,"$595,648,799","28,582"
"Non-film campaigns (kept, not added to a film)","$29,395,173","3,026"
Total kept,"$625,043,972","31,608"


**Left out**

Item,Spend,Buy rows
Deleted buy,"$73,431,792","5,073"
Budget not submitted,"$12,515,956",561
Fee buy,"$2,054,197","1,300"
Budget submitted,"$453,539",86


---
**Notes on the numbers**
- Revenue films with no PVOD date are kept in `revenue_film_daily` but are not in the premium view until a date is added in 05_Review_Titles.
- Media spend whose product code is not mapped yet is kept with link `unmapped`; map it in 05_Review_Titles and run this notebook again.